# Guardrails-AI: Validators and on_fail [Security - Module 02, Notebook 06]

> **MLCourse - Agentic AI - Production Security**

Notebook `05` showed NeMo Guardrails, whose unit of thought is the
**conversational flow**. This notebook covers
[Guardrails-AI](https://github.com/guardrails-ai/guardrails), whose unit of
thought is much smaller and, for most applications, much more immediately
useful: the **validator**.

A validator is one check over one value. Guardrails-AI's contribution is not
the checks themselves -- you already wrote checks by hand in notebook `01` --
but the **framing around what happens when a check fails**.

In notebook `01` every failure did the same thing: return a reason and stop.
Fail closed, always. That is a sound default, but it is only one of several
sensible responses. Sometimes you want to repair the value. Sometimes you want
to drop the bad field and keep the rest. Sometimes you want to ask the model to
try again. Guardrails-AI names these policies, calls them **`on_fail` actions**,
and lets you pick per validator.

### What you will learn

1. Writing a validator with `@register_validator`.
2. Composing several validators into one `Guard`.
3. The four `on_fail` behaviours: `EXCEPTION`, `FIX`, `FILTER`, `REASK`.
4. Validating **structured output** against a Pydantic model.
5. Where each `on_fail` policy is appropriate -- and where it is dangerous.

### Key takeaways

- The valuable idea is **`on_fail` as an explicit policy**, not the validators.
- `FIX` silently changes data; that is sometimes right and often a hidden bug.
- `REASK` costs a second LLM call and may still fail -- bound it.
- For safety-critical checks, `EXCEPTION` (fail closed) remains correct.

### 1. Setup

Two practical notes before we start.

**Telemetry.** Guardrails-AI ships with OpenTelemetry export switched on. In an
offline or restricted environment that produces a wall of connection-failure
warnings that has nothing to do with your code. We turn it off explicitly with
`settings.disable_tracing = True`.

**Validators live on a Hub.** Guardrails-AI 0.11 bundles the *framework* but
almost no ready-made validators -- those are installed separately from the
Guardrails Hub (`guardrails hub install hub://guardrails/toxic_language`), which
needs network access and a hub token. That is worth knowing before you plan
around it.

It also makes this notebook better. We write our own validators with
`@register_validator`, which is the part of the API you will actually use in
production anyway, since domain rules are by definition domain-specific. The
`on_fail` machinery -- the genuinely valuable part -- works identically for
custom and hub validators.

### Setup: environment, telemetry, versions


In [ ]:
import os
import json
import warnings
import importlib.metadata as md
from pathlib import Path
from dotenv import load_dotenv

warnings.filterwarnings("ignore")

def find_env(depth: int = 8):
    """Walk upward until we find 03_agentic_ai/.env (the track's env file)."""
    p = Path.cwd()
    for _ in range(depth):
        candidate = p / "03_agentic_ai" / ".env"
        if candidate.is_file():
            return candidate
        p = p.parent
    return None

ENV_PATH = find_env()
load_dotenv(ENV_PATH, override=False)

from guardrails import Guard, OnFailAction, settings
# Guardrails-AI exports traces by default; silence it so the notebook output
# shows our results rather than network retry warnings.
settings.disable_tracing = True

from guardrails.validators import (
    Validator, PassResult, FailResult, register_validator,
)

print("Module 02 / Notebook 06: Guardrails-AI")
print(f"env file     : {ENV_PATH}")
print(f"guardrails-ai: {md.version('guardrails-ai')}")
print(f"tracing      : disabled")
print("\nAvailable on_fail actions:")
for name in ["EXCEPTION", "FIX", "FILTER", "REASK", "REFRAIN", "NOOP", "FIX_REASK"]:
    print(f"  OnFailAction.{name:10s} = {getattr(OnFailAction, name)!r}")


### 2. Writing a validator

A validator subclasses `Validator` and implements `validate(value, metadata)`,
returning either `PassResult()` or `FailResult(...)`.

The important field is `fix_value`. A `FailResult` may carry a *repaired*
version of the value. This is what makes the `FIX` policy possible: the
validator does not merely detect a problem, it proposes the correction. If you
never intend to use `FIX`, you can omit it.

`@register_validator` gives the validator a name so it can be referenced in
configs and serialised.

Below: a redaction validator, closely related to the `InputSafetyGuard` you
wrote by hand in notebook `01`, but now with a repair path attached.

### A validator that detects and can repair leaked secrets


In [ ]:
import re

SECRET_PATTERNS = [
    (re.compile(r"\bsk-[A-Za-z0-9]{8,}\b"), "[REDACTED_API_KEY]"),
    (re.compile(r"\b\d{3}-\d{2}-\d{4}\b"), "[REDACTED_SSN]"),
    (re.compile(r"\b\d{4}[ -]?\d{4}[ -]?\d{4}[ -]?\d{4}\b"), "[REDACTED_CARD]"),
]

@register_validator(name="no-secrets", data_type="string")
class NoSecrets(Validator):
    """Fails when the text contains a secret; offers a redacted fix_value."""

    def validate(self, value: str, metadata: dict) -> "ValidationResult":
        found, repaired = [], value
        for pattern, replacement in SECRET_PATTERNS:
            if pattern.search(repaired):
                found.append(replacement)
                repaired = pattern.sub(replacement, repaired)
        if found:
            return FailResult(
                error_message=f"secret(s) detected: {', '.join(found)}",
                fix_value=repaired,          # <- enables the FIX policy
            )
        return PassResult()

# Try it directly, outside any Guard, to see the raw result object.
v = NoSecrets()
for sample in ["totally clean text", "my key is sk-abc123def456"]:
    result = v.validate(sample, {})
    kind = type(result).__name__
    detail = getattr(result, "error_message", "")
    fix = getattr(result, "fix_value", None)
    print(f"{kind:12s} {sample!r}")
    if detail:
        print(f"             reason   : {detail}")
        print(f"             fix_value: {fix!r}")


### 3. The four `on_fail` behaviours

This is the heart of the library. The *same* validator behaves in four
different ways depending on the policy you attach at construction time.

| Policy | What happens on failure | Use when |
|---|---|---|
| `EXCEPTION` | raises `ValidationError` | the failure is a real error; safety-critical checks |
| `FIX` | substitutes `fix_value` | the repair is unambiguous and lossless enough |
| `FILTER` | removes the offending value | one bad field shouldn't sink the whole response |
| `REASK` | calls the LLM again with the error | the model can plausibly do better on a retry |

Note the syntax: the policy is passed to the **validator's constructor**, not to
`Guard.use()`:

```python
Guard().use(NoSecrets(on_fail=OnFailAction.FIX))   # correct
Guard().use(NoSecrets, on_fail=OnFailAction.FIX)   # TypeError
```

Let's run one input through all four.

### The same validator under four different policies


In [ ]:
LEAKY = "Here is the key you asked for: sk-abc123def456 - keep it safe."

print("=== One validator, four on_fail policies ===")
print(f"\ninput: {LEAKY!r}\n")

# --- EXCEPTION: fail closed, exactly like notebook 01 ---
guard_exc = Guard().use(NoSecrets(on_fail=OnFailAction.EXCEPTION))
try:
    guard_exc.validate(LEAKY)
    print("EXCEPTION -> (no error raised)")
except Exception as e:
    print(f"EXCEPTION -> raised {type(e).__name__}: {e}")

# --- FIX: substitute the repaired value and carry on ---
guard_fix = Guard().use(NoSecrets(on_fail=OnFailAction.FIX))
r_fix = guard_fix.validate(LEAKY)
print(f"FIX       -> passed={r_fix.validation_passed}  output={r_fix.validated_output!r}")

# --- FILTER: drop the offending value entirely ---
guard_filter = Guard().use(NoSecrets(on_fail=OnFailAction.FILTER))
r_filter = guard_filter.validate(LEAKY)
print(f"FILTER    -> passed={r_filter.validation_passed}  output={r_filter.validated_output!r}")

# --- A clean input passes every policy unchanged ---
print(f"\nclean input through the FIX guard -> "
      f"{guard_fix.validate('nothing sensitive here').validated_output!r}")


### Read the `FIX` output closely

`FIX` returned the string with the key replaced by `[REDACTED_API_KEY]`. The
guard reports `validation_passed=True`.

Pause on that. **The check failed, and the guard says it passed.** That is not a
bug -- `FIX` means "this failure was handled" -- but it has a serious
consequence: if you only ever look at `validation_passed`, a `FIX` policy makes
leaks invisible. The data was silently modified on its way through.

That is fine for cosmetic normalisation (trimming whitespace, clamping a number
into range). It is dangerous for anything you would want to know about. If a
secret was in your model output, you want an *alert*, not a quiet substitution.

`FILTER` is blunter and, in some ways, more honest: the value becomes `None`.
Downstream code that assumes a string will crash rather than proceed on
half-sanitised data.

**Rule of thumb:** use `FIX` when the repair is the point. Use `EXCEPTION` when
the failure is the point.

### 4. Composing validators

Real checks come in groups. `Guard().use(...)` runs several validators over
the same value, and each one carries its own policy. This is the composition
that notebook `01`'s `GuardrailPipeline` did by hand in a `for` loop -- the
difference is that here each check independently decides its own failure
behaviour.

Below we combine three:

- `NoSecrets` with `FIX` -- redact rather than reject,
- `MaxWords` with `EXCEPTION` -- a length breach is a real error,
- `NoProfanity` with `FIX` -- mask and continue.

### More validators, then compose them


In [ ]:
@register_validator(name="max-words", data_type="string")
class MaxWords(Validator):
    """Fails when the text exceeds a word budget; fix_value truncates."""

    def __init__(self, limit: int = 30, **kwargs):
        super().__init__(limit=limit, **kwargs)
        self.limit = limit

    def validate(self, value: str, metadata: dict):
        words = value.split()
        if len(words) > self.limit:
            return FailResult(
                error_message=f"{len(words)} words exceeds limit of {self.limit}",
                fix_value=" ".join(words[: self.limit]),
            )
        return PassResult()


BANNED = {"idiot", "moron", "stupid"}

@register_validator(name="no-profanity", data_type="string")
class NoProfanity(Validator):
    """Fails on banned words; fix_value masks them."""

    def validate(self, value: str, metadata: dict):
        hits = [w for w in BANNED if w in value.lower()]
        if hits:
            repaired = value
            for w in hits:
                repaired = re.sub(w, "*" * len(w), repaired, flags=re.IGNORECASE)
            return FailResult(
                error_message=f"banned words: {', '.join(hits)}",
                fix_value=repaired,
            )
        return PassResult()


composed = Guard().use(
    NoSecrets(on_fail=OnFailAction.FIX),
    NoProfanity(on_fail=OnFailAction.FIX),
    MaxWords(limit=25, on_fail=OnFailAction.EXCEPTION),
)

print("=== Composed guard: NoSecrets(FIX) + NoProfanity(FIX) + MaxWords(EXCEPTION) ===\n")

samples = [
    ("clean",            "Your order ships on Tuesday and arrives by Friday."),
    ("secret+profanity", "That was a stupid mistake, here is sk-abc123def456 again."),
    ("too long",         "word " * 40),
]

for label, text in samples:
    try:
        out = composed.validate(text).validated_output
        print(f"[OK    ] ({label})")
        print(f"         -> {out[:90]!r}")
    except Exception as e:
        print(f"[RAISED] ({label}) {type(e).__name__}: {str(e)[:90]}")
    print()


Notice the ordering effect in the second sample: the `FIX` validators ran and
rewrote the text, and the result carried on to the next validator. Validators
compose as a **pipeline over a mutating value**, not as independent parallel
checks. If a `FIX` validator changes the length of the text, a downstream
length validator sees the *changed* text.

That is convenient, but it means **order matters** and it is not always obvious
from reading the config. This is a genuine debuggability cost compared to the
explicit loop you wrote in notebook `01`, where you could print the value at
every step.

### 5. Structured output validation

Notebook `01` validated structured output with plain Pydantic, which enforced
types, ranges and one custom business rule. Guardrails-AI extends that by
letting you attach validators -- with their `on_fail` policies -- to individual
**fields** of a Pydantic model, and then parse raw model output through it with
`Guard.for_pydantic(...)`.

The gain over plain Pydantic is precisely the per-field failure policy: a bad
`confidence` can raise, while a leaky `answer` gets redacted, in one pass.

### Field-level validators on a Pydantic model


In [ ]:
from pydantic import BaseModel, Field

@register_validator(name="min-confidence", data_type="float")
class MinConfidence(Validator):
    """A too-low confidence should be surfaced, not silently patched."""

    def __init__(self, threshold: float = 0.5, **kwargs):
        super().__init__(threshold=threshold, **kwargs)
        self.threshold = threshold

    def validate(self, value: float, metadata: dict):
        if value < self.threshold:
            return FailResult(
                error_message=f"confidence {value} below threshold {self.threshold}",
                fix_value=self.threshold,
            )
        return PassResult()


class SupportResponse(BaseModel):
    """Compare with SupportResponse in notebook 01 -- same idea, per-field policy."""
    answer: str = Field(
        json_schema_extra={"validators": [NoSecrets(on_fail=OnFailAction.FIX)]}
    )
    confidence: float = Field(
        json_schema_extra={"validators": [MinConfidence(0.5, on_fail=OnFailAction.EXCEPTION)]}
    )
    requires_human: bool = False


struct_guard = Guard.for_pydantic(SupportResponse)

print("=== Structured output validation ===\n")

payloads = [
    ("valid",           {"answer": "Refunds take 5 days.", "confidence": 0.92, "requires_human": False}),
    ("leaky answer",    {"answer": "Use key sk-abc123def456.", "confidence": 0.88, "requires_human": False}),
    ("low confidence",  {"answer": "Maybe try restarting.", "confidence": 0.10, "requires_human": True}),
]

for label, payload in payloads:
    try:
        out = struct_guard.parse(json.dumps(payload)).validated_output
        print(f"[OK    ] ({label}) -> {out}")
    except Exception as e:
        print(f"[RAISED] ({label}) {type(e).__name__}: {str(e)[:80]}")
    print()


The middle case is the one to study. The `answer` field failed its validator but
was **repaired**, so the object parsed successfully with the key redacted. The
last case raised, because `MinConfidence` was configured to fail closed.

One model, two fields, two different failure policies, one call. Doing this by
hand is entirely possible -- it is a `try`/`except` and a couple of `if`
statements -- but on a schema with fifteen fields the declarative version stays
readable where the hand-rolled version does not.

### 6. `REASK`: letting the model fix its own output

`REASK` is the policy with no hand-rolled equivalent in notebook `01`, and the
one that justifies the library for generation-time constraints.

When validation fails, Guardrails-AI constructs a new prompt containing the
original request, the invalid output, and the validator's `error_message`, then
calls the LLM again. The model gets to see *why* it was rejected.

We wire this to Groq (falling back to a local Ollama model if there is no key)
via a plain callable. Guardrails-AI accepts any function taking `messages` and
returning a string -- there is no need for a special integration.

Watch the call counter: it proves the retry actually happened.

### Pick a backend: Groq primary, local Ollama fallback


In [ ]:
import requests

def ollama_up(url="http://localhost:11434/api/tags", timeout=3) -> bool:
    try:
        return requests.get(url, timeout=timeout).status_code == 200
    except Exception:
        return False

HAS_GROQ = bool(os.getenv("GROQ_API_KEY"))

if HAS_GROQ:
    from langchain_groq import ChatGroq
    llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0)
    BACKEND = "Groq / qwen/qwen3.8-27b"
    REASON = "GROQ_API_KEY found -> using Groq"
elif ollama_up():
    from langchain_ollama import ChatOllama
    llm = ChatOllama(model="llama3.1:8b", temperature=0)
    BACKEND = "Ollama / llama3.1:8b"
    REASON = "no Groq key, Ollama is running -> using a local model"
else:
    raise RuntimeError(
        "No LLM backend. Set GROQ_API_KEY in 03_agentic_ai/.env, or run Ollama."
    )

print(f"Decision: {REASON}")
print(f"Backend : {BACKEND}")


### REASK in action


In [ ]:
call_log = []

def call_llm(messages=None, **kwargs) -> str:
    """A plain callable is all Guardrails-AI needs."""
    prompt = "\n".join(m["content"] for m in messages)
    call_log.append(prompt)
    return llm.invoke(prompt).content

reask_guard = Guard().use(MaxWords(limit=12, on_fail=OnFailAction.REASK))

print("=== REASK ===\n")
print("Asking for ~40 words, but the validator allows at most 12.\n")

result = reask_guard(
    call_llm,
    messages=[{"role": "user",
               "content": "Describe the water cycle in about 40 words."}],
    num_reasks=1,
)

print(f"LLM calls made   : {len(call_log)}")
print(f"validation passed: {result.validation_passed}")
print(f"final output     : {result.validated_output!r}")
print(f"final word count : {len(str(result.validated_output).split())}")

print("\n--- what the model saw on the retry (truncated) ---")
if len(call_log) > 1:
    print(call_log[1][:500])


Two LLM calls: the first produced an over-long answer, the validator rejected
it, and Guardrails-AI re-prompted with the error message included. The second
answer satisfied the constraint.

**Be clear-eyed about the cost.** `REASK` doubled latency and doubled token
spend for this request. And it is not guaranteed to work -- the model may fail
the constraint again. `num_reasks` bounds the attempts; when they are exhausted,
`validation_passed` comes back `False` and you must still handle it. `REASK` is
a best-effort improvement, never a guarantee.

For a hard constraint, prefer to make the constraint structural: ask for JSON
with a `max_length`, or truncate deterministically. Retrying a model until it
complies is the expensive way to get a property you could have enforced for
free.

### Pitfalls

- **`FIX` reports success.** `validation_passed=True` after a `FIX` means "the
  failure was handled", not "nothing was wrong". Log fixes, or you will never
  learn that your model leaks keys.
- **Validators mutate in sequence.** A `FIX` earlier in `use` changes what
  later validators see. Order is significant and easy to get wrong.
- **`REASK` costs a full extra generation** and can still fail. Bound it with
  `num_reasks` and always check `validation_passed`.
- **`on_fail` belongs on the validator constructor**, not on `Guard.use()`.
- **Hub validators need network + a token.** Guardrails-AI 0.11 bundles the
  framework, not the validator library. Plan for custom validators, or for the
  `guardrails hub install` step in your build.
- **Telemetry is on by default.** Set `settings.disable_tracing = True` in
  restricted environments or drown in retry warnings.

### Summary


In [ ]:
print("=== Notebook 06 Summary ===\n")
print(f"  backend    : {BACKEND}")
print("  validator  : subclass Validator -> PassResult / FailResult(fix_value=...)")
print("  compose    : Guard().use(...), each with its own policy")
print("  structured : Guard.for_pydantic(Model) for per-field policies")
print()
print("  on_fail policies:")
print("    EXCEPTION -> fail closed          (safety-critical checks)")
print("    FIX       -> substitute fix_value (repair is the point; logs a silent change)")
print("    FILTER    -> drop the value       (one bad field shouldn't sink the response)")
print("    REASK     -> re-prompt the LLM    (+1 call, best-effort, may still fail)")
print()
print("  The real contribution: failure POLICY is explicit and per-check.")
print("  The real risk        : FIX hides problems behind a passing result.")
print("\nNext: 07_llama_guard.ipynb -- a model, not a library, as the safety gate.")
